# Baseline: кредитный скоринг

Один CSV в `data/credit_scoring_dataset.csv` -> 8 признаков -> логистическая регрессия (`numpy`) -> метрики.

## 1. Загрузка одного датасета

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

# Путь к папке с данными (запускать ноутбук из корня проекта)
DATA_DIR = Path.cwd() / "data"

# Загружаем единый датасет кредитного скоринга
df = pd.read_csv(DATA_DIR / "credit_scoring_dataset.csv")

print("shape:", df.shape)
df.head()

shape: (500, 10)


,credit_id,age,monthly_income,employment_years,loan_amount,loan_term_months,interest_rate,past_due_30d,inquiries_6m,target
0,1,25,27364.67,2.4,487842.22,36,35.39,2,3,1
1,2,55,15287.57,17.3,507153.78,24,35.42,1,1,0
2,3,50,16924.78,18.7,722465.61,48,36.95,1,2,0
3,4,40,59589.61,4.0,197260.50,24,23.06,2,5,0
4,5,40,18022.97,12.5,419248.47,24,30.15,0,0,0


In [4]:
data_dict = pd.read_csv(DATA_DIR / "data_dictionary.csv")
print("Справочник полей:")
data_dict

FileNotFoundError: [Errno 2] No such file or directory: '/Users/dmitriisoroka/Documents/projects/2026-04-25 Hackaton/Hackaton/data/data_dictionary.csv'

## 2. Подготовка признаков

In [ ]:
# Список признаков, которые подаем в модель
feature_cols = [
    "age",
    "monthly_income",
    "employment_years",
    "loan_amount",
    "loan_term_months",
    "interest_rate",
    "past_due_30d",
    "inquiries_6m",
]

# Формируем матрицу признаков X и целевой вектор y
X = df[feature_cols].to_numpy(dtype=np.float64)
y = df["target"].to_numpy(dtype=np.float64)

print("X shape:", X.shape)
print("bad rate:", round(float(y.mean()), 3))

X shape: (500, 8)
bad rate: 0.304


## 3. Логистическая регрессия и метрики

In [ ]:
# Фиксируем random seed для воспроизводимости
rng = np.random.default_rng(42)
n = len(y)

# Делим данные на train/test (80/20)
ix = rng.permutation(n)
cut = int(0.8 * n)
tr, te = ix[:cut], ix[cut:]
X_tr, X_te = X[tr], X[te]
y_tr, y_te = y[tr], y[te]

# Стандартизация: считаем параметры только на train
mu, sd = X_tr.mean(0), X_tr.std(0) + 1e-9
X_tr = (X_tr - mu) / sd
X_te = (X_te - mu) / sd

# Добавляем столбец единиц для свободного члена (bias)
Xm = np.column_stack([np.ones(len(X_tr)), X_tr])
w = np.zeros(Xm.shape[1])

# Обучаем логистическую регрессию градиентным спуском
lr = 0.15
for _ in range(3500):
    z = np.clip(Xm @ w, -30, 30)
    p = 1.0 / (1.0 + np.exp(-z))
    w -= lr * Xm.T @ (p - y_tr) / len(y_tr)

# Предсказания на тесте
Xm_te = np.column_stack([np.ones(len(X_te)), X_te])
z_te = np.clip(Xm_te @ w, -30, 30)
p_te = 1.0 / (1.0 + np.exp(-z_te))
pred = (p_te >= 0.5).astype(int)


# Метрики качества

def roc_auc(y_true, s):
    y_true = y_true.astype(bool)
    pos, neg = s[y_true], s[~y_true]
    return (np.sum(pos[:, None] > neg) + 0.5 * np.sum(pos[:, None] == neg)) / (len(pos) * len(neg))


def pr_auc(y_true, s):
    o = np.argsort(-s)
    yt = y_true[o].astype(int)
    tp = np.cumsum(yt)
    fp = np.cumsum(1 - yt)
    prec = tp / np.maximum(tp + fp, 1)
    return float((prec * yt).sum() / max(int(yt.sum()), 1))


print("ROC-AUC:", round(roc_auc(y_te, p_te), 4))
print("PR-AUC :", round(pr_auc(y_te, p_te), 4))
print("Accuracy @0.5:", round((pred == y_te).mean(), 4))

ROC-AUC: 0.7625
PR-AUC : 0.6396
Accuracy @0.5: 0.78


## 4. Пример использования модели на новой заявке

In [ ]:
# Пример одной новой заявки (вручную заданные данные)
new_application = {
    "age": 29,
    "monthly_income": 32000.0,
    "employment_years": 3.5,
    "loan_amount": 650000.0,
    "loan_term_months": 48,
    "interest_rate": 33.0,
    "past_due_30d": 2,
    "inquiries_6m": 4,
}

# Приводим вход к тому же формату, что использовался при обучении
new_df = pd.DataFrame([new_application])
new_X = new_df[feature_cols].to_numpy(dtype=np.float64)

# Используем те же параметры стандартизации, что и при обучении
new_X_scaled = (new_X - mu) / sd
new_Xm = np.column_stack([np.ones(len(new_X_scaled)), new_X_scaled])

# Считаем вероятность класса bad и переводим в метку
p_bad = 1.0 / (1.0 + np.exp(-np.clip(new_Xm @ w, -30, 30)))
label = "bad" if p_bad[0] >= 0.5 else "good"

print("Новая заявка:")
print(new_df)
print("\nРезультат скоринга:")
print(f"P(bad) = {p_bad[0]:.4f}")
print(f"P(good)= {1 - p_bad[0]:.4f}")
print(f"Предсказанный класс: {label}")

Новая заявка:
   age  monthly_income  employment_years  loan_amount  loan_term_months  \
0   29         32000.0               3.5     650000.0                48   

   interest_rate  past_due_30d  inquiries_6m  
0           33.0             2             4  

Результат скоринга:
P(bad) = 0.4420
P(good)= 0.5580
Предсказанный класс: good
